# STAC AI Workshop NYC · October 21, 2026
## Harbor Metals: direct tools, shared MCP tools and permissions
**Section 1 companion — return to the permission example in Section 7.**

Run cells from top to bottom. The default walkthrough uses explicit illustrative requests, not a live model. Actual Python retrieval, arithmetic and permission checks execute locally. Optional cells use OpenAI or a real local MCP connection.

**Look for:** who requests a step, who executes it, and who can authorize release. All company data is fictional.

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01b_harbor_tools_mcp_permissions.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01b_harbor_tools_mcp_permissions.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

## Open the lab: local JupyterLab or Google Colab
**Local:** open this notebook beside `harbor_tools.py` and `harbor_mcp_server.py`.

**Browser alternative:**
1. Open [Google Colab](https://colab.research.google.com/). Choose **File → Open notebook → GitHub**.
2. Paste `https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/01b_harbor_tools_mcp_permissions.ipynb`.
3. Open the notebook; choose **File → Save a copy in Drive** if you want your own editable copy.
4. Upload `harbor_tools.py` and `harbor_mcp_server.py` using the Files sidebar, or run the setup cell below.

Colab runs Python on a Google-hosted machine. Only fictional data is used here. API keys are optional; do not save them in notebook cells.

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:5** | `if "google.colab" in sys.modules:` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
# Colab-only setup: fetch the two small companion files from the public repository.
# Local JupyterLab users already have them beside the notebook.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    from urllib.request import urlretrieve
    base = "https://raw.githubusercontent.com/mfrdixon/STAC_AI_Bootcamp/main/notebooks/"
    for filename in ("harbor_tools.py", "harbor_mcp_server.py"):
        if not Path(filename).exists():
            urlretrieve(base + filename, filename)
    print("Companion files ready.")

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `from harbor_tools import search_documents, calculate_leverage, dispatch, GRANTS, AUDIT, RELEASES` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
from harbor_tools import search_documents, calculate_leverage, dispatch, GRANTS, AUDIT, RELEASES
import json

def show(value):
    print(json.dumps(value, indent=2))
QUESTION = "Does Harbor Metals have comfortable covenant headroom?"
print(QUESTION)

## 1. Try retrieval yourself
**Input:** a search query. **Output:** source IDs and passage text. This small keyword retriever is deliberately transparent. It does not perform semantic ranking.
Run the search. Notice the covenant references an annex that is not in this first result.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:4** | `passages = search_documents(query)` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
query = "Harbor Metals covenant financial figures"
print("QUERY:", query)
passages = search_documents(query)
for passage in passages:
    print(passage["source"], "→", passage["text"])

## 2. Follow the reference, then calculate
Change the query to find the annex. The calculator does not search documents: the caller supplies its arguments. Cash 15 minus restricted cash 5 gives eligible cash 10.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:2** | `for passage in search_documents("Harbor Metals annex"):` | Execute search and return passages; inspect IDs and text, not only scores. |
| **C04:4** | `result = calculate_leverage(debt=75, eligible_cash=10, ebitda=20)` | Execute the calculator with these arguments; inspect the cash definition. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
for passage in search_documents("Harbor Metals annex"):
    print(passage["source"], "→", passage["text"])
result = calculate_leverage(debt=75, eligible_cash=10, ebitda=20)
show(result)
print("Headroom:", round(3.5-result["leverage"], 2), "times")

**Interpretation:** current leverage is 3.25×, below 3.5×. “Comfortable” requires an agreed buffer and stress assumptions; compliance alone is insufficient. With EBITDA 17, leverage becomes 3.82×.

## 3. Fixed workflow versus agent-selected tools
**Fixed:** code specifies search → annex search → calculator → model interpretation.
**Agent:** the model receives tool definitions, asks for a call, receives its result, then chooses its next request. The application executes each call.

<!-- WORKSHOP-CODE-GUIDE C05 -->
### Code cell C05 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C05:3** | `initial = search_documents("Harbor Metals financial figures covenant")` | Execute search and return passages; inspect IDs and text, not only scores. |
| **C05:5** | `calculation = calculate_leverage(75, 10, 20)` | Execute the calculator with these arguments; inspect the cash definition. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C05 · code cell
# A fixed workflow: these steps are specified by application code.
initial = search_documents("Harbor Metals financial figures covenant")
annex = search_documents("Harbor Metals annex")
calculation = calculate_leverage(75, 10, 20)
context = {"question": QUESTION, "sources": initial + annex, "calculation": calculation}
# This is the context supplied to a generator for interpretation.
for source in context["sources"]:
    print(source["source"], "→", source["text"])
show(context["calculation"])

## 4. Optional: let a GPT model choose the calls
Install `openai` if needed. Set `RUN_LIVE=True` only to make billed API calls. The key is requested privately if absent from the environment; it is not printed or saved.

**Watch the printed trace:** model request → Python execution → returned result. Only search and calculation are exposed. No MCP server is needed.

<!-- WORKSHOP-CODE-GUIDE C06 -->
### Code cell C06 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C06:2** | `RUN_LIVE = False` | This switch controls an optional live, model-download or training path. Leave it False for the default exercise. |
| **C06:10** | `tool("search_documents", "Search Harbor Metals documents. Follow annex references.",` | Execute search and return passages; inspect IDs and text, not only scores. |
| **C06:12** | `tool("calculate_leverage", "Compute net leverage using evidence-supported eligible cash.",` | Execute the calculator with these arguments; inspect the cash definition. |
| **C06:24** | `response = client.responses.create(model=MODEL, instructions=instructions,` | Send a real OpenAI request when the optional live path is enabled. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C06 · code cell
RUN_LIVE = False
MODEL = "gpt-5"  # Choose a tool-capable model available to your account.

def tool(name, description, properties):
    return {"type":"function", "name":name, "description":description,
            "parameters":{"type":"object", "properties":properties,
                          "required":list(properties), "additionalProperties":False}, "strict":True}
TOOLS = [
    tool("search_documents", "Search Harbor Metals documents. Follow annex references.",
         {"query":{"type":"string"}}),
    tool("calculate_leverage", "Compute net leverage using evidence-supported eligible cash.",
         {k:{"type":"number"} for k in ("debt","eligible_cash","ebitda")})
]
if RUN_LIVE:
    import os, getpass
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: "))
    history = [{"role":"user", "content":QUESTION}]
    instructions = ("Use Harbor Metals documents and calculator. Follow referenced annexes. "
                    "Do not equate covenant compliance with comfortable headroom. Cite sources.")
    completed_tools = []
    for step in range(8):
        response = client.responses.create(model=MODEL, instructions=instructions,
                                           input=history, tools=TOOLS, tool_choice="auto")
        history.extend(response.output)
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print("FINAL ANSWER:", response.output_text)
            if "calculate_leverage" not in completed_tools:
                print("NOT VERIFIED: no calculator call completed; review required.")
            break
        for call in calls:
            args = json.loads(call.arguments)
            print("MODEL REQUEST:", call.name, args)
            output = dispatch("research", call.name, args)
            completed_tools.append(call.name)
            print("APPLICATION RETURNS:"); show(output)
            history.append({"type":"function_call_output", "call_id":call.call_id,
                            "output":json.dumps(output)})
    else:
        print("Stopped at the application step limit; no final answer established.")
else:
    print("Live calls off. The local walkthrough above runs without a key.")

## 5. Why add MCP? Reuse these same tools
```text
Credit assistant ───┐
Monitoring agent ───┼── MCP server ── Harbor Metals functions and documents
Analyst notebook ───┘
```
Direct calling suits one application. MCP standardizes discovery and calling when several compatible clients need the same service. MCP does not determine the workflow or grant permission.

The supplied `harbor_mcp_server.py` wraps the same functions. Optional local connection below uses the MCP Python SDK v1 (`pip install "mcp>=1.12,<2"`). It launches a server subprocess: no public endpoint or cloud deployment is required. The client below is Python, not a live agent.

<!-- WORKSHOP-CODE-GUIDE C07 -->
### Code cell C07 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C07:2** | `RUN_MCP = False` | This switch controls an optional live, model-download or training path. Leave it False for the default exercise. |
| **C07:19** | `result = await session.call_tool("search_documents", {"query":"Harbor Metals annex"})` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C07 · code cell
RUN_MCP = False
if RUN_MCP:
    import os, sys
    from pathlib import Path
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def shared_client_example(workflow):
        # Demo principal is fixed by launcher config, not a model-supplied argument.
        params = StdioServerParameters(command=sys.executable,
            args=[str(Path("harbor_mcp_server.py").resolve())],
            env={**os.environ, "HARBOR_DEMO_PRINCIPAL":"research"})
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                print(workflow, "discovers:", [t.name for t in tools.tools])
                result = await session.call_tool("search_documents", {"query":"Harbor Metals annex"})
                print(result.content)
                denied = await session.call_tool("release_brief", {"brief":"Release Harbor brief"})
                print("Release denied:", denied.isError)
                assert denied.isError
    await shared_client_example("Credit assistant")
    await shared_client_example("Monitoring workflow")
else:
    print("Optional MCP connection off; permission exercise below still runs locally.")

## 6. Permissions: a plausible answer is not authority to release
The research role can search and calculate. The reviewer role can also release a brief. **The application/server checks the grant before the side effect.** Hiding a tool from the model is not sufficient enforcement.

Here the launcher/caller is trusted and identities are local role names. In production derive identity and grants from authenticated credentials; never let a caller select `reviewer` as an unverified parameter. The in-memory audit is a teaching record, not a durable audit system.

<!-- WORKSHOP-CODE-GUIDE C08 -->
### Code cell C08 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C08:5** | `dispatch("research", "release_brief", {"brief":brief})` | Check the caller grant before executing the named function. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C08 · code cell
brief = "Harbor Metals: current leverage 3.25x; comfort remains unresolved."
AUDIT.clear(); RELEASES.clear()
try:
    dispatch("research", "release_brief", {"brief":brief})
except PermissionError as error:
    print("BLOCKED:", error)
assert not RELEASES
show(dispatch("reviewer", "release_brief", {"brief":brief}))
assert len(RELEASES) == 1

# Revoke authority, then retry the identical request.
GRANTS["reviewer"].remove("release_brief")
try:
    try:
        dispatch("reviewer", "release_brief", {"brief":brief})
    except PermissionError as error:
        print("BLOCKED AFTER REVOCATION:", error)
    assert len(RELEASES) == 1
finally:
    GRANTS["reviewer"].add("release_brief")
show(AUDIT)

## Check your understanding
1. What selected the next step in the fixed workflow?
2. Does MCP authorize release?
3. Does a 3.25× ratio prove comfortable headroom?

**Answers:** application code; no—the server/application enforces grants; no—an agreed buffer and downside assessment are still needed.

**Later governance session:** revisit identity, revocation, evidence sufficiency, accountable approval and durable audit trails. This example verifies tool-level permissions, not the factual correctness of a brief.

Sources: [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling), [MCP Python SDK v1](https://github.com/modelcontextprotocol/python-sdk/tree/v1.x), [OpenAI MCP integration](https://developers.openai.com/api/docs/guides/tools-connectors-mcp).

## Answer checks · default configuration

- Fixed flow: application code chooses both searches and calculation. Direct calling needs no MCP.
- Filing: debt 75, reported cash 15, EBITDA 20. Annex: restricted cash 5. Eligible cash is 10; leverage is **3.25×**, headroom to 3.5× is **0.25×**.
- Compliance alone does not define “comfortable”; downside EBITDA 17 gives approximately **3.82×**.
- Research release: **denied**. Reviewer release: **one simulated release**. After revocation: **denied**, with no second release.
- MCP exposes tools through a standard interface; the server/application enforces permission.
- Optional live or MCP-disabled cells correctly report disabled. Do not expect a new provider response when their switches are False.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.